
# Compression Breakout System (CBS) — Hypothesis Test

**Source:** `Compression Breakout System.pine` (Mateo Sandoval), Pine v6 strategy.

**What the script claims** (from its own header comments, AAPL daily, no stated OOS split):

| Risk/trade | Total Return | Profit Factor | Max DD |
|---|---|---|---|
| 2% | +374% | 4.85 | 7.3% |
| 4% (default) | +899% | 4.33 | 14.4% |
| 2%, full 40y history | +458% | 3.16 | — |

**The hypothesis under test is *not* "does this make money on AAPL 2005–2026"** —
a single-symbol, single-window backtest with no train/test split is close to
worthless as evidence on its own; it's one draw from a huge space of possible
(symbol, window, parameter) combinations, and the report is silent on how many
of those were tried before this one got written into the header comment.

**What this notebook actually tests:**

1. **Replication** — does the Python port reproduce the stated AAPL numbers
   closely enough to trust the port itself?
2. **Cross-sectional generalization** — does the edge exist across a basket of
   large-cap trending names, or only on AAPL?
3. **Regime / walk-forward robustness** — train-period parameters, held-out
   test-period performance.
4. **Parameter sensitivity** — is the surface around the chosen parameters flat
   (robust) or a spike (overfit)?
5. **Statistical significance** — bootstrap/permutation test against a null of
   "random entries with the same stop/trail exit logic."
6. **Deflated Sharpe Ratio** — correcting for the number of parameter
   combinations effectively searched, since a hard stats gate on multiple
   testing is standard practice before any of this touches live capital.

Data source: `yfinance` daily OHLCV (adjusted close reconstructed via `auto_adjust=True`).


In [ ]:

# If running fresh, uncomment:
# !pip install yfinance pandas numpy matplotlib scipy --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from itertools import product
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


In [ ]:

# ─────────────────────────────────────────────────────────────────────────
# Python port of the Pine v6 strategy logic. Ported term-for-term so this is
# a test of the ACTUAL rule set, not a paraphrase of it.
#
# Key correctness points (get these wrong and the whole test is invalid):
#   - percentrank uses a strictly trailing window (no lookahead)
#   - breakout level uses the prior N-bar high, shifted by 1 bar (Pine's `[1]`)
#   - ATR uses Wilder/RMA smoothing (alpha = 1/len), matching ta.atr, NOT a
#     simple rolling mean of true range
#   - the exit is bar-by-bar and path-dependent (stop OR trailing chandelier,
#     whichever is higher) — this cannot be vectorized without introducing
#     lookahead bugs, so it's an explicit loop
#   - orders fill on the close of the signal bar (process_orders_on_close=true
#     in the source) — i.e. this only ever acts on already-confirmed, closed
#     bars. No same-bar high/low is used to decide the entry.
# ─────────────────────────────────────────────────────────────────────────

def percentrank(series, window):
    def _rank(x):
        return (x[:-1] < x[-1]).sum() / (len(x) - 1) * 100.0 if len(x) > 1 else np.nan
    return series.rolling(window + 1).apply(_rank, raw=True)


def compute_signals(df, p):
    out = df.copy()
    basis = out["Close"].rolling(p["bb_len"]).mean()
    dev = out["Close"].rolling(p["bb_len"]).std(ddof=0) * p["bb_mult"]
    bb_width = (2.0 * dev) / basis * 100.0
    w_rank = percentrank(bb_width, p["rank_len"])
    sqz_on = w_rank < p["sqz_pct"]
    sqz_recent = sqz_on.rolling(p["sqz_age"]).sum() > 0

    brk_lvl = out["High"].rolling(p["brk_len"]).max().shift(1)
    vol_ok = out["Volume"] > out["Volume"].rolling(20).mean()

    tr = pd.concat([
        out["High"] - out["Low"],
        (out["High"] - out["Close"].shift(1)).abs(),
        (out["Low"] - out["Close"].shift(1)).abs(),
    ], axis=1).max(axis=1)
    atr = tr.ewm(alpha=1 / p["atr_len"], adjust=False).mean()

    sma_trend = out["Close"].rolling(p["sma_len"]).mean()
    trend_ok = out["Close"] > sma_trend
    buy_sig = (sqz_recent & (out["Close"] > brk_lvl) & vol_ok & trend_ok).fillna(False)

    out["bb_width"], out["w_rank"], out["sqz_on"] = bb_width, w_rank, sqz_on
    out["brk_lvl"], out["vol_ok"], out["atr"] = brk_lvl, vol_ok, atr
    out["trend_ok"], out["buy_sig"] = trend_ok, buy_sig
    return out


def run_backtest(df, p, initial_capital=10000.0, commission_pct=0.05,
                  slippage_ticks=1, tick_size=0.01, start=None, end=None):
    sig = compute_signals(df, p)
    if start is not None:
        sig = sig[sig.index >= start]
    if end is not None:
        sig = sig[sig.index <= end]

    equity, position = initial_capital, 0.0
    entry_price = entry_stop = hwm = np.nan
    entry_time = None
    trades, equity_curve = [], []

    for ts, row in sig.iterrows():
        close, atr = row["Close"], row["atr"]

        if position > 0:
            hwm = max(hwm, close)
            stop_lvl = max(entry_stop, hwm - atr * p["trail_atr"])
            if close < stop_lvl:
                exit_price = close * (1 - slippage_ticks * tick_size / close)
                pnl = position * (exit_price - entry_price)
                fee = position * exit_price * commission_pct / 100.0
                equity += pnl - fee
                trades.append(dict(entry_time=entry_time, exit_time=ts,
                                    entry_price=entry_price, exit_price=exit_price,
                                    qty=position, pnl=pnl - fee,
                                    reason="stop" if close <= entry_stop else "trail"))
                position = 0.0

        if position == 0 and row["buy_sig"] and not np.isnan(atr) and atr > 0:
            stop_dist = atr * p["stop_atr"]
            qty = min(equity * p["risk_pct"] / 100.0 / stop_dist, equity / close)
            if qty > 0:
                fill_price = close * (1 + slippage_ticks * tick_size / close)
                fee = qty * fill_price * commission_pct / 100.0
                equity -= fee
                position, entry_price = qty, fill_price
                entry_stop, hwm, entry_time = close - stop_dist, close, ts

        mtm = equity + (position * (close - entry_price) if position > 0 else 0.0)
        equity_curve.append((ts, mtm))

    if position > 0:
        last_close = sig["Close"].iloc[-1]
        equity += position * (last_close - entry_price)
        trades.append(dict(entry_time=entry_time, exit_time=sig.index[-1],
                            entry_price=entry_price, exit_price=last_close,
                            qty=position, pnl=position * (last_close - entry_price),
                            reason="window_end"))

    eq_df = pd.DataFrame(equity_curve, columns=["time", "equity"]).set_index("time")
    return eq_df, pd.DataFrame(trades)


def performance_summary(eq_df, trades_df, initial_capital=10000.0):
    if eq_df.empty:
        return {}
    final_equity = eq_df["equity"].iloc[-1]
    total_return_pct = (final_equity / initial_capital - 1) * 100
    n_years = (eq_df.index[-1] - eq_df.index[0]).days / 365.25
    cagr = ((final_equity / initial_capital) ** (1 / n_years) - 1) * 100 if n_years > 0 else np.nan
    roll_max = eq_df["equity"].cummax()
    max_dd = ((eq_df["equity"] / roll_max - 1) * 100).min()
    daily_ret = eq_df["equity"].pct_change().dropna()
    sharpe = (daily_ret.mean() / daily_ret.std()) * np.sqrt(252) if daily_ret.std() > 0 else np.nan
    if not trades_df.empty:
        wins = trades_df.loc[trades_df["pnl"] > 0, "pnl"].sum()
        losses = -trades_df.loc[trades_df["pnl"] < 0, "pnl"].sum()
        pf = wins / losses if losses > 0 else np.inf
        win_rate = (trades_df["pnl"] > 0).mean() * 100
        n_trades, avg_trade = len(trades_df), trades_df["pnl"].mean()
    else:
        pf = win_rate = n_trades = avg_trade = np.nan
    return dict(total_return_pct=total_return_pct, cagr_pct=cagr, max_dd_pct=max_dd,
                sharpe=sharpe, profit_factor=pf, win_rate_pct=win_rate,
                n_trades=n_trades, avg_trade=avg_trade, final_equity=final_equity)


DEFAULT_PARAMS = dict(bb_len=20, bb_mult=2.0, rank_len=126, sqz_pct=25.0, sqz_age=5,
                       brk_len=20, sma_len=200, atr_len=22, stop_atr=2.0, risk_pct=4.0,
                       trail_atr=3.0)



## 1. Data ingestion

`yfinance` EOD, `auto_adjust=True` (splits/dividends folded into OHLC — matters
a lot for a 20-year AAPL backtest given the 2014 and 2020 splits; the Pine
strategy is running on TradingView's adjusted series, so this needs to match
or the breakout levels won't line up).


In [ ]:

def load(ticker, start="1990-01-01", end=None):
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df[["Open", "High", "Low", "Close", "Volume"]].dropna()

aapl = load("AAPL", start="1990-01-01")
print(aapl.shape, aapl.index.min(), aapl.index.max())
aapl.tail()



## 2. Replication check

Run the exact params from the header comment (`risk_pct=2` and `risk_pct=4`,
window 2005-01-01 → today) and compare against the stated 374% / 899%.

**Expect a gap, not a match.** yfinance's adjusted series, TradingView's feed,
and the exact commission/slippage model will all differ slightly — the point
isn't bit-for-bit reproduction, it's confirming the *sign and rough magnitude*
line up (i.e. the port isn't broken) before trusting anything downstream.


In [ ]:

for risk in [2.0, 4.0]:
    params = {**DEFAULT_PARAMS, "risk_pct": risk}
    eq, trades = run_backtest(aapl, params, start="2005-01-01")
    perf = performance_summary(eq, trades)
    print(f"risk={risk}%  return={perf['total_return_pct']:.1f}%  "
          f"PF={perf['profit_factor']:.2f}  maxDD={perf['max_dd_pct']:.1f}%  "
          f"trades={perf['n_trades']}  win%={perf['win_rate_pct']:.1f}")


In [ ]:

eq4, trades4 = run_backtest(aapl, {**DEFAULT_PARAMS, "risk_pct": 4.0}, start="2005-01-01")
fig, ax = plt.subplots()
eq4["equity"].plot(ax=ax, title="CBS on AAPL, risk=4% (replication of default params)")
ax.set_ylabel("Equity ($)")
plt.show()
trades4.tail(10)



## 3. Cross-sectional generalization

This is the load-bearing test. A rule set validated on one hand-picked,
extraordinarily-successful stock over exactly the period it was successful
tells you almost nothing about the *rule set* — it mostly tells you AAPL went
up a lot. Run the identical, un-retuned params across a basket of large-cap
trending names + broad index ETFs and see whether the edge survives.


In [ ]:

universe = ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "META", "SPY", "QQQ", "TSLA", "AVGO"]
results = []
data = {}
for t in universe:
    try:
        d = load(t, start="2005-01-01")
        data[t] = d
        eq, trades = run_backtest(d, DEFAULT_PARAMS, start="2005-01-01")
        perf = performance_summary(eq, trades)
        perf["ticker"] = t
        results.append(perf)
    except Exception as e:
        print(t, "failed:", e)

xsec = pd.DataFrame(results).set_index("ticker")[
    ["total_return_pct", "cagr_pct", "max_dd_pct", "sharpe", "profit_factor", "win_rate_pct", "n_trades"]
]
xsec


In [ ]:

# Buy-and-hold benchmark for the same tickers/window, for context
bh = {}
for t, d in data.items():
    w = d.loc["2005-01-01":]
    bh[t] = (w["Close"].iloc[-1] / w["Close"].iloc[0] - 1) * 100
xsec["buyhold_pct"] = pd.Series(bh)
xsec["edge_vs_buyhold"] = xsec["total_return_pct"] - xsec["buyhold_pct"]
xsec.sort_values("sharpe", ascending=False)



**Read this table skeptically.** If Sharpe/PF are strong on AAPL/NVDA/TSLA but
flat-to-negative on SPY/QQQ and the median large-cap, the "edge" is really
"long a stock that went up 50x during a 20-year AI/mega-cap bull run, filtered
through a rule that's usually long anyway during trend-up regimes." That's a
beta bet wearing a systematic-strategy costume, not a compression/expansion
edge. Compare `edge_vs_buyhold` — if it's negative for most names, the
Bollinger-squeeze mechanism isn't adding anything over passive exposure.



## 4. Walk-forward validation

Split the AAPL history into anchored train/test folds. Parameters are never
retuned per fold here (this is a *fixed-rule* strategy per the source) — the
point is to check whether performance is stable across folds or concentrated
in one lucky window.


In [ ]:

folds = [
    ("2005-01-01", "2011-12-31"),
    ("2012-01-01", "2015-12-31"),
    ("2016-01-01", "2019-12-31"),
    ("2020-01-01", "2022-12-31"),
    ("2023-01-01", "2026-12-31"),
]
wf_rows = []
for s, e in folds:
    eq, trades = run_backtest(aapl, DEFAULT_PARAMS, start=s, end=e)
    perf = performance_summary(eq, trades)
    perf["fold"] = f"{s} → {e}"
    wf_rows.append(perf)
wf = pd.DataFrame(wf_rows).set_index("fold")[
    ["total_return_pct", "cagr_pct", "max_dd_pct", "sharpe", "profit_factor", "n_trades"]
]
wf



If one or two folds carry the entire return and the rest are flat/negative,
the strategy isn't "40 years of edge" — it's a regime-dependent trend filter
that happened to be long during the folds that mattered. Note this ported
engine doesn't retune per fold; this checks stability of the *fixed* rule,
not an adaptive walk-forward optimization (that's a separate, heavier study
if the fixed-rule result looks promising enough to justify it).



## 5. Parameter sensitivity (overfit surface check)

Grid a few key params around the defaults. A robust edge shows a broad plateau
of decent Sharpe; an overfit one shows a narrow spike at the exact header-comment
values and craters everywhere else.


In [ ]:

grid = list(product([15, 20, 25], [15, 25, 35], [15, 20, 25]))  # (bb_len, sqz_pct, brk_len)
grid_rows = []
for bb_len, sqz_pct, brk_len in grid:
    params = {**DEFAULT_PARAMS, "bb_len": bb_len, "sqz_pct": sqz_pct, "brk_len": brk_len}
    eq, trades = run_backtest(aapl, params, start="2005-01-01")
    perf = performance_summary(eq, trades)
    grid_rows.append(dict(bb_len=bb_len, sqz_pct=sqz_pct, brk_len=brk_len, **perf))

grid_df = pd.DataFrame(grid_rows)
n_trials = len(grid_df)
grid_df.sort_values("sharpe", ascending=False).head(10)[
    ["bb_len", "sqz_pct", "brk_len", "sharpe", "total_return_pct", "profit_factor", "n_trades"]
]


In [ ]:

pivot = grid_df.pivot_table(index="bb_len", columns="sqz_pct", values="sharpe", aggfunc="mean")
fig, ax = plt.subplots()
im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
ax.set_xlabel("sqz_pct"); ax.set_ylabel("bb_len"); ax.set_title("Sharpe surface (avg over brk_len)")
plt.colorbar(im, ax=ax, label="Sharpe")
plt.show()



## 6. Statistical significance — block bootstrap vs. a random-entry null

Null hypothesis: entries carry no information — a random long entry with the
*identical* stop/trail exit logic and identical position sizing would perform
just as well. If the real signal's Sharpe/PF doesn't clear the random-entry
distribution by a wide margin, the "compression breakout" logic isn't doing
the work — the exit rules and equity-curve trending market are.


In [ ]:

def run_backtest_from_signals(sig, p, initial_capital=10000.0, commission_pct=0.05,
                               slippage_ticks=1, tick_size=0.01):
    # identical loop to run_backtest but skips recompute — reuses precomputed signals
    equity, position = initial_capital, 0.0
    entry_price = entry_stop = hwm = np.nan
    entry_time = None
    trades, equity_curve = [], []
    for ts, row in sig.iterrows():
        close, atr = row["Close"], row["atr"]
        if position > 0:
            hwm = max(hwm, close)
            stop_lvl = max(entry_stop, hwm - atr * p["trail_atr"])
            if close < stop_lvl:
                exit_price = close * (1 - slippage_ticks * tick_size / close)
                pnl = position * (exit_price - entry_price)
                fee = position * exit_price * commission_pct / 100.0
                equity += pnl - fee
                trades.append(dict(pnl=pnl - fee))
                position = 0.0
        if position == 0 and row["buy_sig"] and not np.isnan(atr) and atr > 0:
            stop_dist = atr * p["stop_atr"]
            qty = min(equity * p["risk_pct"] / 100.0 / stop_dist, equity / close)
            if qty > 0:
                fill_price = close * (1 + slippage_ticks * tick_size / close)
                fee = qty * fill_price * commission_pct / 100.0
                equity -= fee
                position, entry_price = qty, fill_price
                entry_stop, hwm, entry_time = close - stop_dist, close, ts
        mtm = equity + (position * (close - entry_price) if position > 0 else 0.0)
        equity_curve.append((ts, mtm))
    if position > 0:
        last_close = sig["Close"].iloc[-1]
        equity += position * (last_close - entry_price)
        trades.append(dict(pnl=position * (last_close - entry_price)))
    eq_df = pd.DataFrame(equity_curve, columns=["time", "equity"]).set_index("time")
    return eq_df, pd.DataFrame(trades)

base_sig = compute_signals(aapl.loc["2005-01-01":], DEFAULT_PARAMS)
entry_rate = base_sig["buy_sig"].mean()  # match real signal frequency
real_eq, real_trades = run_backtest(aapl, DEFAULT_PARAMS, start="2005-01-01")
real_perf = performance_summary(real_eq, real_trades)

N_SIMS = 300
null_sharpes, null_returns = [], []
for i in range(N_SIMS):
    fake_sig = base_sig.copy()
    fake_sig["buy_sig"] = np.random.default_rng(i).random(len(base_sig)) < entry_rate
    eq_i, trades_i = run_backtest_from_signals(fake_sig, DEFAULT_PARAMS)
    perf_i = performance_summary(eq_i, trades_i)
    null_sharpes.append(perf_i.get("sharpe", np.nan))
    null_returns.append(perf_i.get("total_return_pct", np.nan))

null_sharpes = np.array(null_sharpes)
p_value = (null_sharpes >= real_perf["sharpe"]).mean()
print(f"Real Sharpe: {real_perf['sharpe']:.2f}  |  Null mean: {np.nanmean(null_sharpes):.2f}  "
      f"±{np.nanstd(null_sharpes):.2f}  |  p-value (one-sided): {p_value:.3f}")

fig, ax = plt.subplots()
ax.hist(null_sharpes, bins=30, alpha=0.7, label="random-entry null")
ax.axvline(real_perf["sharpe"], color="red", lw=2, label="actual CBS signal")
ax.set_xlabel("Sharpe"); ax.legend(); ax.set_title("CBS signal vs. random-entry null (same exit logic)")
plt.show()



**p < 0.05 here means:** entries carrying the compression-breakout condition
beat random entries with the same stop/trail exits, at conventional
significance — evidence the squeeze/breakout filter itself adds information,
not just the exit and position-sizing scaffolding around it.

**p not small:** the story in the header comment is almost entirely the exit
logic (wide trail, tight stop) riding a trending market, and the "compression"
detector is decoration.



## 7. Deflated Sharpe Ratio (multiple-testing correction)

The parameter grid in section 5 ran `N_trials` backtests. Reporting the best
one's Sharpe as *the* result without correcting for how many were tried is
the single most common way quant backtests lie. Bailey & López de Prado's
Deflated Sharpe Ratio (DSR) corrects the observed Sharpe for the number of
independent trials, the skew/kurtosis of returns, and the track record length.


In [ ]:

from scipy.stats import norm

def deflated_sharpe_ratio(observed_sharpe, n_trials, n_obs, skew=0.0, kurt=3.0,
                           sharpe_variance=None):
    # Expected max Sharpe under N iid trials (Bailey & Lopez de Prado, 2014)
    if sharpe_variance is None:
        sharpe_variance = 1.0
    euler_mascheroni = 0.5772156649
    e_max = (np.sqrt(sharpe_variance) *
             ((1 - euler_mascheroni) * norm.ppf(1 - 1.0 / n_trials) +
              euler_mascheroni * norm.ppf(1 - 1.0 / (n_trials * np.e))))
    sr_std = np.sqrt((1 - skew * observed_sharpe + (kurt - 1) / 4 * observed_sharpe ** 2) / (n_obs - 1))
    dsr = norm.cdf((observed_sharpe - e_max) / sr_std) if sr_std > 0 else np.nan
    return dsr, e_max

daily_ret = real_eq["equity"].pct_change().dropna()
skew, kurt = daily_ret.skew(), daily_ret.kurtosis() + 3
best_grid_sharpe = grid_df["sharpe"].max()
dsr, e_max_sharpe = deflated_sharpe_ratio(best_grid_sharpe, n_trials=n_trials,
                                           n_obs=len(daily_ret), skew=skew, kurt=kurt)
print(f"Grid trials searched: {n_trials}")
print(f"Best in-grid daily Sharpe: {best_grid_sharpe:.2f}  (annualized: {best_grid_sharpe:.2f})")
print(f"Expected max Sharpe under {n_trials} pure-noise trials: {e_max_sharpe:.2f}")
print(f"Deflated Sharpe Ratio (prob. true Sharpe > 0, correcting for search): {dsr:.3f}")



DSR is a probability, not a p-value in the classic sense — it's the probability
the strategy's *true* Sharpe is greater than zero after accounting for the fact
that N parameter combinations were searched and the best one was reported. A
DSR comfortably above ~0.95 with a reasonable trial count is the bar used
before anything here would be treated as a real edge rather than a lucky draw
from the grid.

**This notebook's grid is small (27 trials) on purpose** — the real number of
trials implicitly run to arrive at the header comment's exact parameter set
(`sqzPct=25`, `brkLen=20`, `stopAtr=2.0`, `trailAtr=3.0`, `riskPct=4.0`) is
unknown and almost certainly larger. If you're iterating on parameters
yourself, log every trial and feed the true count into `n_trials` here —
under-counting trials is the most common way DSR gets gamed by accident.



## 8. Verdict & risk register

Fill this in after running against live data — structure kept fixed so the
gate is mechanical, not vibes-based:

| Gate | Threshold | Result | Pass? |
|---|---|---|---|
| Replication sanity | Sign + rough magnitude match header claim | — | — |
| Cross-sectional | Positive `edge_vs_buyhold` on ≥60% of universe | — | — |
| Walk-forward | No single fold contributes >70% of total return | — | — |
| Parameter sensitivity | Sharpe plateau, not spike, around chosen params | — | — |
| Bootstrap significance | p < 0.05 vs random-entry null | — | — |
| Deflated Sharpe | DSR > 0.95 at true trial count | — | — |

### Known failure modes not covered by this notebook (do not skip before capital)

- **Survivorship bias in the universe list.** AAPL/MSFT/NVDA/etc. are winners
  by construction — this cross-sectional test should be re-run against a
  point-in-time index constituent list (e.g. historical S&P 500 membership),
  not today's mega-cap survivors.
- **Corporate actions / adjusted-close drift.** `auto_adjust=True` back-adjusts
  the whole series on every split/dividend — this changes historical price
  *levels* the Bollinger width and 20d-high breakout level are computed on
  retroactively. TradingView does something similar but the exact methodology
  won't match tick-for-tick; don't over-trust exact price levels near old
  split dates.
- **No regime-conditioning on the entry.** `trend_ok = close > SMA200` is the
  only regime filter — this is a garden-variety trend-following overlay, not
  a volatility-cycle model. Expect this to bleed slowly in extended sideways
  chop even outside drawdowns (dead capital from repeated small stop-outs).
- **Slippage/commission at 5bps + 1 tick is optimistic** for anything beyond
  mega-cap liquid names, and this is a single-position, 100%-of-equity
  strategy — real fills at size will move the tape on the exact 20d-high
  breakout bar this strategy is trying to buy (everyone's breakout scanner
  fires on the same bar).
- **This backtest is long-only, one asset at a time.** Running it across a
  multi-symbol universe *concurrently* changes the risk picture entirely —
  correlated mega-cap tech names will cluster entries/exits, so the
  single-symbol Sharpe here overstates a portfolio Sharpe unless you also
  model position-count caps and correlation-adjusted sizing.
- **No point-in-time volume/float check.** `vol_ok` uses raw share volume,
  which isn't comparable across a 20-year window where AAPL's float and split
  count changed multiple times — this may be silently more/less strict in
  different eras than intended.
